# 06 · Transformer safety-text model (NTSB, optional ASRS) — TPU or GPU

Fine-tunes DistilRoBERTa for the same task as the shipped TF-IDF model — fatal-outcome triage from probable-cause text with
outcome words masked — on the same **temporal split** (train 2008–2015, validate 2016, test 2017–2022), so the numbers are
directly comparable. Optional: if you export ASRS reports (CSV, ≤10,000 per download) into `data/public/asrs/`, the last
cell trains an anomaly-category classifier on them. ASRS narratives are self-reported and **not verified by NASA**.

In [ ]:
# Get the repo, regenerate the synthetic corpus (deterministic, ~5 s) and fetch the public datasets (checksummed)
!git clone -q https://github.com/<you>/aeronexus-ai.git /content/aeronexus-ai || true
%cd /content/aeronexus-ai/backend
!pip -q install -r requirements.txt
!python scripts/generate_data.py --train 4000
!python scripts/fetch_public_data.py

In [ ]:
import importlib, subprocess, sys
try:
    import jax; tpu = any(d.platform == "tpu" for d in jax.devices())
except Exception:
    tpu = False
if tpu:
    import torch; v = torch.__version__.split("+")[0]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"torch_xla[tpu]=={v}", "-f",
                    "https://storage.googleapis.com/libtpu-releases/index.html"], check=False)
!pip -q install transformers datasets accelerate evaluate
sys.path.insert(0, ".")
from aeronexus.safety.ntsb import load_events, split
df = load_events(); tr, va, te = split(df); print(len(tr), len(va), len(te), "TPU:", tpu)

In [ ]:
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import roc_auc_score, average_precision_score
M = "distilroberta-base"; tok = AutoTokenizer.from_pretrained(M)
enc = lambda d: Dataset.from_dict({"text": d.text_masked.tolist(), "label": d.fatal.tolist()}).map(
    lambda b: tok(b["text"], truncation=True, max_length=128, padding="max_length"), batched=True)
dtr, dva, dte = enc(tr), enc(va), enc(te)
pos_w = (1 - tr.fatal.mean()) / tr.fatal.mean()
model = AutoModelForSequenceClassification.from_pretrained(M, num_labels=2)
args = TrainingArguments("safety-out", per_device_train_batch_size=32, per_device_eval_batch_size=64, num_train_epochs=2,
                         learning_rate=3e-5, eval_strategy="epoch", save_strategy="no", report_to=[], seed=7)
trainer = Trainer(model=model, args=args, train_dataset=dtr, eval_dataset=dva)
trainer.train()
logits = trainer.predict(dte).predictions; p = np.exp(logits)[:, 1] / np.exp(logits).sum(1)
print("test AUROC", roc_auc_score(te.fatal, p), "PR-AUC", average_precision_score(te.fatal, p))
print("compare with the shipped TF-IDF model in data/reports/safety_ntsb.json")
trainer.save_model("/content/export/safety-transformer"); tok.save_pretrained("/content/export/safety-transformer")

In [ ]:
# The API serves the calibrated TF-IDF safety model; retrain it here so VS Code gets the matching artifact + report.
!python scripts/train_public_models.py --only safety
ARTIFACTS = ["models/safety_ntsb.joblib", "data/reports/safety_ntsb.json", "/content/export/safety-transformer"]

In [ ]:
# Save artifacts to Google Drive (survives the Colab session; avoids flaky browser downloads of big files).
# In VS Code: python scripts/import_models.py <the downloaded aeronexus_export.zip or Drive folder>
import hashlib, json, os, platform, shutil, sys, time
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/aeronexus/export"
os.makedirs(OUT, exist_ok=True)
for item in ARTIFACTS:                                   # files or folders produced above
    dst = os.path.join(OUT, os.path.basename(item.rstrip("/")))
    (shutil.copytree(item, dst, dirs_exist_ok=True) if os.path.isdir(item) else shutil.copy2(item, dst))
def sha(p):
    h = hashlib.sha256(); h.update(open(p, "rb").read()); return h.hexdigest()
import sklearn, numpy
man_path = os.path.join(OUT, "manifest.json")
man = json.load(open(man_path)) if os.path.exists(man_path) else {"artifacts": {}}
for root, _, fs in os.walk(OUT):
    for f in fs:
        if f != "manifest.json":
            p = os.path.join(root, f); man["artifacts"][os.path.relpath(p, OUT)] = sha(p)
man.update({"exported_at": time.strftime("%Y-%m-%dT%H:%M:%S"), "python": platform.python_version(),
            "versions": {"scikit-learn": sklearn.__version__, "numpy": numpy.__version__}})
json.dump(man, open(man_path, "w"), indent=1)
shutil.make_archive("/content/drive/MyDrive/aeronexus/aeronexus_export", "zip", OUT)
print("saved to Drive: MyDrive/aeronexus/aeronexus_export.zip  →", sorted(man["artifacts"])[:12])

In [ ]:
# Optional: ASRS anomaly categories from your own export (column names vary by export; adjust if needed)
import glob, pandas as pd
files = glob.glob("data/public/asrs/*.csv")
if files:
    a = pd.concat(pd.read_csv(f, low_memory=False, skiprows=[1]) for f in files)
    text = next(c for c in a.columns if "Narrative" in c); lab = next(c for c in a.columns if "Anomaly" in c)
    a = a[[text, lab]].dropna(); a["y"] = a[lab].str.split(";").str[0].str.split(" ").str[0]
    print(a["y"].value_counts().head(10))
    # → reuse the Trainer above with num_labels = number of top categories
else:
    print("No ASRS export found — see https://asrs.arc.nasa.gov/search/database.html")